## Concept 7 — ReAct Prompts (Reason + Act)

### What is it?
ReAct = **Re**asoning + **Act**ing. The LLM writes its thoughts before each action,
creating a transparent Thought → Action → Observation loop.

### Pipeline
```
Question
  → Thought: what do I know / what do I need?
  → Action:  tool_name(args)
  → Observation: tool result
  → [repeat if needed]
  → Final Answer
```

### Limitation (what Concept 8 fixes)
Prompt logic is hardcoded — difficult to reuse parts across different agents/tasks.
→ Concept 8 introduces Partial Prompts for dynamic, modular template composition.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from PromptUtils import get_llm
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain.agents import create_react_agent, AgentExecutor

llm = get_llm(temperature=0.0)
print('LLM ready')

### Step 1 — Define Tools
ReAct agents use tools. The docstring is critical — the LLM reads it to decide which tool to use.

In [ ]:
@tool
def search(query: str) -> str:
    """Search for factual information about a topic."""
    # Mocked knowledge base — replace with real search (DuckDuckGo, Tavily, etc.)
    facts = {
        'python creator':      'Python was created by Guido van Rossum in 1991.',
        'langchain':           'LangChain is a framework for building LLM-powered applications, created by Harrison Chase.',
        'openai':              'OpenAI was founded in 2015 by Sam Altman, Elon Musk, and others. Sam Altman is the CEO.',
        'gpt-4':               'GPT-4 is a large multimodal model by OpenAI released in March 2023.',
        'transformer':         'The Transformer architecture was introduced in the paper Attention Is All You Need (2017) by Vaswani et al.',
    }
    query_lower = query.lower()
    for key, fact in facts.items():
        if key in query_lower:
            return fact
    return f'No specific result found for: {query}'

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. Input must be a valid Python expression."""
    try:
        result = eval(expression, {'__builtins__': {}}, {})
        return str(result)
    except Exception as e:
        return f'Error: {e}'

tools = [search, calculate]
print(f'Tools: {[t.name for t in tools]}')

### Step 2 — The ReAct Prompt Template
The prompt instructs the LLM to follow the Thought/Action/Observation format exactly.

In [ ]:
react_template = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format EXACTLY:

Question: the input question you must answer
Thought: you should always think about what to do next
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}"""

react_prompt = PromptTemplate.from_template(react_template)
print('ReAct prompt defined')

### Step 3 — Create the ReAct Agent
`create_react_agent` wires the LLM + tools + ReAct prompt together.
`AgentExecutor` runs the loop until `Final Answer` is produced.

In [ ]:
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=react_prompt
)

executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,          # prints Thought/Action/Observation each step
    max_iterations=5,      # safety limit
    handle_parsing_errors=True,
)
print('Agent ready')

### Step 4 — See Thought/Action/Observation in Action
Watch the LLM reason step-by-step before and after each tool call.

In [ ]:
result = executor.invoke({'input': 'Who created Python and in what year? Also, what is 1991 + 33?'})
print(f'\nFinal Answer: {result["output"]}')

### Step 5 — Multi-Hop Reasoning
Question that requires two search steps — each Observation informs the next Thought.

In [ ]:
result = executor.invoke({
    'input': 'Who is the CEO of the company that made GPT-4?'
})
print(f'\nFinal Answer: {result["output"]}')

### Full Function

In [ ]:
def react_agent(query: str) -> str:
    result = executor.invoke({'input': query})
    return result['output']

queries = [
    'What year was the Transformer paper published?',
    'What is 2017 + 8?',
]

print('ReAct Agent Results:')
for q in queries:
    print(f'\nQ: {q}')
    print(f'A: {react_agent(q)}')
    print('-' * 40)